### Libraries and constants

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils import get_null_info

In [2]:
# --- CONSTANTS ---
RANDOM_SEED = 42

### Data import

In [3]:
links_raw = pd.read_csv(Path('data') / 'links.csv')
links_raw

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


In [4]:
movies_raw = pd.read_csv(Path('data') / 'movies.csv')
movies_raw

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [5]:
ratings_raw = pd.read_csv(Path('data') / 'ratings.csv')
ratings_raw

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [6]:
tags_raw = pd.read_csv(Path('data') / 'tags.csv')
tags_raw

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


In [7]:
ratings_raw.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [8]:
# I don't see why we'd need timestamp
ratings_raw = ratings_raw.drop(columns=['timestamp'])

In [9]:
duplicates_rating = ratings_raw[ratings_raw.duplicated()]
duplicates_rating

,userId,movieId,rating


In [10]:
ratings_raw.dtypes

userId       int64
movieId      int64
rating     float64
dtype: object

In [11]:
get_null_info(ratings_raw)

No missing values are found in the data_frame


""


In [12]:
ratings_raw['rating'].value_counts()

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

In [13]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

In [14]:
# Tell surprise the rating scale
scale = (ratings_raw.rating.min(), ratings_raw.rating.max())
reader = Reader(rating_scale=scale)

# Hand over ONLY the three columns, in the order (user, item, rating).
# This is our 𝒦 — the list of observed triples, exactly as-is.
data = Dataset.load_from_df(ratings_raw[['userId', 'movieId', 'rating']], reader)

# split into train and test
df_train, df_test = train_test_split(data, test_size=0.2, random_state=RANDOM_SEED, shuffle=True)

# The model. This SVD is literally the thing we derived — biased matrix
# factorization trained by SGD. Each argument is one of the symbols:
algo = SVD(
    n_factors=100,   # k  — how many latent factors (atoms) to keep
    n_epochs=20,     #    — full passes over the ratings
    lr_all=0.005,    # α  — learning rate (your step size)
    reg_all=0.02,    # λ  — regularization strength
    biased=True,
    random_state=RANDOM_SEED,
)

# Train. Under the hood this runs your two update rules, plus the μ + b_u + b_i
# bias terms, looping one triple at a time — the b=1 SGD from your note.
algo.fit(df_train)

# Score on the held-out ratings with RMSE — your held-out metric, in star units.
predictions = algo.test(df_test)
accuracy.rmse(predictions)          # expect ~0.87 on this dataset

RMSE: 0.8807


0.8807462819979623

In [15]:
algo.predict(uid=1, iid=1).est      # user 1's predicted rating for movie 1

4.572613688080741

Actual recommendations — the point of the whole exercise. Predict every movie this user hasn't rated, sort, take the top 10, and decode to titles via movies.csv:

In [16]:
user_id = 1

rated   = ratings_raw.loc[ratings_raw.userId == user_id, 'movieId']
unrated = movies_raw.loc[~movies_raw.movieId.isin(rated), 'movieId']

preds = [(mid, algo.predict(user_id, mid).est) for mid in unrated]
preds.sort(key=lambda x: x[1], reverse=True)

for mid, est in preds[:10]:
    title = movies_raw.loc[movies_raw.movieId == mid, 'title'].values[0]
    print(f"{est:.2f}  {title}")

5.00  Blade Runner (1982)
5.00  Ghost in the Shell (Kôkaku kidôtai) (1995)
5.00  Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
5.00  North by Northwest (1959)
5.00  Casablanca (1942)
5.00  Streetcar Named Desire, A (1951)
5.00  Cinema Paradiso (Nuovo cinema Paradiso) (1989)
5.00  One Flew Over the Cuckoo's Nest (1975)
5.00  Lawrence of Arabia (1962)
5.00  Grand Day Out with Wallace and Gromit, A (1989)


In [17]:
from surprise.model_selection import cross_validate
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8752  0.8711  0.8696  0.8746  0.8754  0.8732  0.0024  
MAE (testset)     0.6709  0.6724  0.6709  0.6708  0.6708  0.6711  0.0006  
Fit time          1.09    1.04    1.03    1.04    1.04    1.05    0.02    
Test time         0.11    0.11    0.12    0.11    0.29    0.15    0.07    


{'test_rmse': array([0.87522336, 0.8710988 , 0.86964908, 0.87462196, 0.875358  ]),
 'test_mae': array([0.67086972, 0.67241444, 0.67088795, 0.67078371, 0.67075061]),
 'fit_time': (1.0863854885101318,
  1.0438354015350342,
  1.0327692031860352,
  1.0430569648742676,
  1.0383055210113525),
 'test_time': (0.1110234260559082,
  0.1110372543334961,
  0.1154317855834961,
  0.10809707641601562,
  0.2881796360015869)}

In [19]:
from surprise.model_selection import GridSearchCV

# Each key is a symbol from your notes; each list is the values to try.
param_grid = {
    'n_factors': [50, 100],      # k — number of latent factors
    'n_epochs':  [20, 30],       #   — passes over the data
    'lr_all':    [0.005, 0.01],  # α — learning rate
    'reg_all':   [0.02, 0.1],    # λ — regularization
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)          # note: full `data`, NOT trainset — it makes its own folds

print(gs.best_score['rmse'])    # the best RMSE it found
print(gs.best_params['rmse'])   # the winning combination

0.8621052519766618
{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [ ]:
best = gs.best_estimator['rmse']
best.fit(data.build_full_trainset())

In [ ]:
from surprise import SVD, SVDpp, NMF
from surprise.model_selection import cross_validate

for Algo in (SVD, SVDpp, NMF):
    result = cross_validate(Algo(), data, measures=['RMSE'], cv=3, verbose=False)
    print(f"{Algo.__name__:8s}  RMSE = {result['test_rmse'].mean():.4f}")

In [ ]:
import pandas as pd
import sympy as sp

def analytical_SVD(M: pd.DataFrame):
    """SVD from scratch via the eigenvalue route:  M = U Σ Vᵀ (thin)."""
    # ---------- Step 1: Make it square if it's not to be able to find eigenvectors ----------
    m, n = M.shape
    M_transpose = M.T
    square_M = pd.DataFrame()

    if m != n:
        if m < n:
            square_M = M @ M_transpose
        else:
            square_M = M_transpose @ M
    else:
        square_M = M

    # ---------- Step 2: Find eigenvalues (using characteristic equation: det(M - lambda*I) = 0) ----------
    # Convert to a sympy Matrix for symbolic linear algebra operations
    M_sym = sp.Matrix(square_M.to_numpy())
    lam = sp.symbols("lambda")
    I = sp.eye(M_sym.shape[0])
    characteristic_eq = (M_sym - lam * I).det()

    # Solve the determinant equation for lambda, sorted
    M_eigenvalues = sorted(np.array(sp.solve(characteristic_eq, lam), dtype=float), reverse=True)

    # ---------- Step 3: Find singular values for SVD ----------
    M_singular_values = np.sqrt(M_eigenvalues)

    # ---------- Step 4: Find the normalized u vectors (eigenvectors of MM^T) ----------
    u_vectors = []
    for val in M_eigenvalues:
        # Plug lambda back into (MM^T - lambda * I)
        homogenous_system = M_sym - val * I

        # Solve for the null space (vectors where system * u = 0)
        null_space = homogenous_system.nullspace()

        for vec in null_space:
            print(vec)
            # Normalize the vector to unit length (divide by its magnitude)
            u_vectors.append(vec / np.linalg.norm(np.array(vec, dtype=float)))

    # ---------- Step 5: Find the v vectors via $\vec{v_i} = \tfrac{1}{\sigma_i}M^T\vec{u_i}$ ----------
    M_mat = sp.Matrix(M.to_numpy())
    v_vectors = []

    for sigma, vec in zip(M_singular_values, u_vectors, strict=True):
        if sigma < 1e-9:  # Avoid division by zero for zero singular values
            continue

        if m < n:
            # We have U, finding V: v = (1/sigma) * M^T * u
            v_vec = (1 / sigma) * (M_mat.T * vec)
            v_vectors.append(v_vec)
        else:
            # We have V, finding U: u = (1/sigma) * M * v
            u_vec = (1 / sigma) * (M_mat * vec)
            v_vectors.append(u_vec)

    # ---------- Step 6: Construct final SVD matrices as standard NumPy arrays ----------
    if m < n:
        U = np.hstack([np.array(u, dtype=float) for u in u_vectors])
        V_T = np.hstack([np.array(v, dtype=float) for v in v_vectors]).T
    else:
        U = np.hstack([np.array(u, dtype=float) for u in v_vectors])
        V_T = np.hstack([np.array(v, dtype=float) for v in u_vectors]).T

    Sigma = np.diag(M_singular_values)

    return U, Sigma, V_T

In [ ]:
# rng = np.random.default_rng(seed=RANDOM_SEED)
# M = rng.integers(low=1, high=100, size=(3, 4))
# M

array([[ 9, 77, 65, 44],
       [43, 86,  9, 70],
       [20, 10, 53, 97]], dtype=int64)

In [78]:
M = pd.DataFrame([
    [1, 1, 0],
    [0, 1, 1],
])

analytical_SVD(M)

2 < 3: $M^T M$ (shape: (3, 2))
Matrix([[-1.00000000000000], [1]])
Matrix([[1.00000000000000], [1]])


(array([[-0.70710678,  0.70710678],
        [ 0.70710678,  0.70710678]]),
 array([[1.        , 0.        ],
        [0.        , 1.73205081]]),
 array([[-0.70710678,  0.        ,  0.70710678],
        [ 0.40824829,  0.81649658,  0.40824829]]))

In [25]:
pd.Series([1,2,3]).mean()

2.0

In [74]:
# Step 1: obtain the sparse pivot grid full of 0s (your initial data where for each user there are values in columns for certain movies and empty cells on the ones they didn't rate)
# Step 2: make a dataframe that only includes existing data (e.g., user_id, movie_id, rating)
# Step 3: make a holdout test
# Step 4: Train

def GD_SVD(u_idx, i_idx, ratings, num_users, num_items,
           latent_factors=100, learning_rate=0.02, regularization=0.02,
           epochs=20, random_seed=42):
    rng = np.random.default_rng(random_seed)
    P = rng.normal(0, 0.1, (num_users, latent_factors))
    Q = rng.normal(0, 0.1, (num_items, latent_factors))
    b_u, b_i = np.zeros(num_users), np.zeros(num_items)
    mu = ratings.mean()

    u_idx, i_idx, ratings = np.asarray(u_idx), np.asarray(i_idx), np.asarray(ratings)
    n = len(ratings)

    for ep in range(epochs):
        for k in rng.permutation(n):              # shuffle every epoch
            u, i, r = u_idx[k], i_idx[k], ratings[k]

            p_u = P[u].copy()                     # snapshot BEFORE updating
            q_i = Q[i].copy()
            e = r - (mu + b_u[u] + b_i[i] + p_u @ q_i)

            P[u]  += learning_rate * (e * q_i - regularization * p_u)
            Q[i]  += learning_rate * (e * p_u - regularization * q_i)
            b_u[u] += learning_rate * (e - regularization * b_u[u])
            b_i[i] += learning_rate * (e - regularization * b_i[i])

    return mu, b_u, b_i, P, Q

def predict(df, mu, b_u, b_i, P, Q):
    u, i = df['u'].to_numpy(), df['i'].to_numpy()
    dot = np.sum(P[u] * Q[i], axis=1)      # row-wise p_u · q_i, no dense grid
    return mu + b_u[u] + b_i[i] + dot

In [75]:
user_ids = ratings_raw['userId'].unique()
item_ids = ratings_raw['movieId'].unique()
user_to_idx = {raw: pos for pos, raw in enumerate(user_ids)}   # name -> seat number
item_to_idx = {raw: pos for pos, raw in enumerate(item_ids)}
num_users, num_items = len(user_ids), len(item_ids)

ratings = ratings_raw.copy()
ratings['u'] = ratings['userId'].map(user_to_idx)     # positional columns
ratings['i'] = ratings['movieId'].map(item_to_idx)

In [77]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

# data preparation
df_train, df_test = train_test_split(ratings, test_size=0.2, random_state=RANDOM_SEED)

# train
mu, b_u, b_i, P, Q = GD_SVD(df_train['u'], df_train['i'], df_train['rating'],
                            num_users, num_items, learning_rate=0.005, epochs=20)

# evaluate
for name, df in [('train', df_train), ('test', df_test)]:
    rmse = root_mean_squared_error(df['rating'], predict(df, mu, b_u, b_i, P, Q))
    print(f"{name}: {rmse:.4f}")

train: 0.6395
test: 0.8847
